# Pythonizing Vincent!

In [20]:
import numpy as np
from typing import Union, Optional


# ------------------------------------------------------------
# 1) Ppi: build the orthogonal matrix V from first-order Pis
# ------------------------------------------------------------

def Ppi(Pi: Union[np.ndarray, list]) -> np.ndarray:
    """
    Python/Numpy port of the R function Ppi(Pi).

    Parameters
    ----------
    Pi : array-like of shape (N,)
        First-order inclusion probabilities, 0 < Pi_i < 1,
        sum(Pi) must be (approximately) an integer.

    Returns
    -------
    V : ndarray of shape (N, n)
        Orthogonal matrix associated to the DSD construction.
    """
    Pi = np.asarray(Pi, dtype=float).ravel()
    N = Pi.size

    # --- Error checks ---
    if N < 2:
        raise ValueError(
            "The sampling designs should be defined on a set of more than "
            "one element. (length(Pi) > 1)"
        )

    if np.any(Pi <= 0) or np.any(Pi >= 1):
        raise ValueError("Pi is not a vector of probabilities (0 < p < 1).")

    sum_pi_rounded = round(Pi.sum(), 9)
    n_int = int(sum_pi_rounded)
    if int(round(sum_pi_rounded, 9)) - sum_pi_rounded != 0:
        raise ValueError(
            "The sum of the first order inclusion probabilities "
            "should be an integer (up to rounding)."
        )

    # --- Main algorithm ---
    s_vals = np.zeros(N, dtype=float)
    c_vals = np.zeros(N, dtype=float)
    alpha = np.zeros(N, dtype=float)

    # kr will store the indices (0-based) where cumulative sum crosses integers
    if n_int <= 0:
        raise ValueError("Sum of Pi must be at least 1.")
    kr = [None] * n_int

    cum_sum = 0.0
    r = 1          # current integer threshold
    r_prev = 0     # last integer that was crossed

    for k in range(N):
        prev_sum = cum_sum
        cum_sum += Pi[k]

        if cum_sum >= r:  # crossed integer r
            if r <= n_int:
                alpha[k] = r - prev_sum
                kr[r - 1] = k   # store 0-based index
                val = np.sqrt((1.0 - Pi[k]) / (1.0 - alpha[k]))
                s_vals[k] = np.round(val, 8)
                r_prev = r
                r += 1
        else:
            denom = (r_prev + 1 - prev_sum)
            val = np.sqrt(Pi[k] / denom)
            s_vals[k] = np.round(val, 15)

        c_vals[k] = np.sqrt(1.0 - s_vals[k] ** 2)

    # Patch: ensure last crossing index corresponds to last unit (like R hack)
    # If some entries of kr are still None, set the last one to N-1.
    if any(x is None for x in kr):
        kr[-1] = N - 1
    # For safety, also replace any remaining None by the last index
    last_index = kr[-1]
    kr = [last_index if x is None else x for x in kr]

    # Use sample size n_int as number of columns (simplified vs. R hack)
    r_prev = n_int

    # --- Build V ---
    V = np.zeros((N, r_prev), dtype=float)
    V[0, 0] = 1.0

    # In R: V[kr[r] + 1, r + 1] = 1 for r in 1:(r_prev-1)
    # Here r_idx corresponds to r-1 in R, and we use 0-based indices.
    if r_prev - 1 != 0:
        for r_idx in range(1, r_prev):
            kpos = kr[r_idx - 1]  # 0-based index for row
            V[kpos + 1, r_idx] = 1.0

    # Apply Givens-like rotations
    for k in range(N - 1):
        L = V[k, :].copy()
        M = V[k + 1, :].copy()
        V[k, :] = s_vals[k] * L - c_vals[k] * M
        V[k + 1, :] = c_vals[k] * L + s_vals[k] * M

    return V


# ------------------------------------------------------------
# 2) DSD sampling (real and complex versions)
# ------------------------------------------------------------

def Drawing_Dsd(
    v: Union[np.ndarray, list],
    s: int = 1,
    B: bool = False,
    seed: Optional[int] = None,
):
    """
    Python/Numpy port of the R function Drawing_Dsd(v, s=1, B=FALSE, seed=NULL).

    Parameters
    ----------
    v : array-like, shape (N, n)
        Matrix of (real or complex) vectors used in DSD.
    s : int, default 1
        Number of samples (replicates).
    B : bool, default False
        If True: return 0/1 indicator vector(s).
        If False: return indices of selected units (1-based, to match R).
    seed : int or None
        Seed for reproducibility.

    Returns
    -------
    If s == 1:
        1D array of length N (if B=True) or selected indices (if B=False).
    If s > 1:
        2D array of shape (N, s) (if B=True) or (n, s) with indices per sample.
    """
    v = np.asarray(v)
    rng = np.random.default_rng(seed)

    if np.iscomplexobj(v):
        return _dsd_sampling_mult_complex(v, s, B, rng)
    else:
        return _dsd_sampling_mult(v, s, B, rng)


# ---------------------- helpers: real case ---------------------- #

def _dsd_sampling_mult(
    v: np.ndarray,
    s: int,
    B: bool,
    rng: np.random.Generator,
):
    v = np.asarray(v, dtype=float)
    if v.ndim == 1:
        v = v[:, None]  # treat as (N,1)

    if s == 1:
        return _dsd_sampling_01_B_C(v, B, rng)
    else:
        samples = [_dsd_sampling_01_B_C(v, B, rng) for _ in range(s)]
        # stack as columns like R's replicate (N x s)
        return np.column_stack(samples)


def _dsd_sampling_01_B_C(
    v: np.ndarray,
    B: bool = True,
    rng: Optional[np.random.Generator] = None,
):
    """
    Real-valued version of .dsd_sampling_01_B_C in R.
    """
    v = np.asarray(v, dtype=float)
    if v.ndim == 1:
        v = v[:, None]

    N, n = v.shape
    echant = np.zeros(N, dtype=int)

    if rng is None:
        rng = np.random.default_rng()
    ref = rng.random(n)

    # Step 1: first element
    w = v.copy()

    pi1 = np.einsum("ij,ij->i", v, v)  # diag(v %*% t(v))
    total = 0.0
    i = -1

    while total < ref[0]:
        i += 1
        if i >= N:
            raise RuntimeError("Sampling failed in first step (real case).")
        total += pi1[i] / n
    echant[i] = 1

    l = v[i, :]
    norm_l = np.sqrt(np.dot(l, l))
    if norm_l == 0:
        raise ValueError("Encountered zero-norm vector in real DSD.")
    e1 = l / norm_l

    # Step 2: remaining n-1 elements
    for j in range(n - 1):
        r = n - (j + 1)
        inter = v @ e1  # (N,)
        pi1 = pi1 - inter * inter
        pi2 = pi1 / r

        total = 0.0
        i = -1
        while total < ref[j + 1]:
            i += 1
            if i >= N:
                raise RuntimeError("Sampling failed in step 2 (real case).")
            total += pi2[i]
        echant[i] = 1

        # Update w and e1 (Gram-Schmidt-like update)
        proj = w @ e1                      # shape (N,)
        w = w - np.outer(proj, e1)         # (N,n)
        L = w[i, :]
        norm_L = np.sqrt(np.dot(L, L))
        if norm_L == 0:
            raise ValueError("Encountered zero-norm vector in real DSD.")
        e1 = L / norm_L

    if B:
        # 0/1 vector -> same as R "echant"
        return echant
    else:
        # indices (1-based, like in R: (1:N)[echant==1])
        return np.nonzero(echant == 1)[0] + 1


# -------------------- helpers: complex case --------------------- #

def _dsd_sampling_mult_complex(
    v: np.ndarray,
    s: int,
    B: bool,
    rng: np.random.Generator,
):
    v = np.asarray(v, dtype=np.complex128)
    if v.ndim == 1:
        v = v[:, None]

    if s == 1:
        return _dsd_sampling_01_B_C_complex(v, B, rng)
    else:
        samples = [_dsd_sampling_01_B_C_complex(v, B, rng) for _ in range(s)]
        return np.column_stack(samples)


def _dsd_sampling_01_B_C_complex(
    v: np.ndarray,
    B: bool = True,
    rng: Optional[np.random.Generator] = None,
):
    """
    Complex-valued version of .dsd_sampling_01_B_C_complex in R.
    """
    v = np.asarray(v, dtype=np.complex128)
    if v.ndim == 1:
        v = v[:, None]

    N, n = v.shape
    echant = np.zeros(N, dtype=int)

    if rng is None:
        rng = np.random.default_rng()
    ref = rng.random(n)

    w = v.copy()

    # pi1 = Re( diag( v %*% t(Conj(v)) ) )
    pi1 = np.real(np.einsum("ij,ij->i", v, np.conjugate(v)))

    if np.any(pi1 < 0) or np.any(pi1 >= 1):
        raise ValueError(
            "The matrix v given as input doesn't suit the expected input "
            "(cf. pgd / periodic_dsd)."
        )

    # Step 1: first element
    total = 0.0
    i = -1

    while total < ref[0]:
        i += 1
        if i >= N:
            raise RuntimeError("Sampling failed in first step (complex case).")
        total += pi1[i] / n
    echant[i] = 1

    M = v[i, :]
    norm_M = np.sqrt(np.real(np.vdot(M, M)))
    if norm_M == 0:
        raise ValueError("Encountered zero-norm vector in complex DSD.")
    e1 = M / norm_M

    # Step 2: remaining n-1 elements
    for j in range(n - 1):
        r = n - (j + 1)

        # inter <- v %*% Conj(e1)
        inter = v @ np.conjugate(e1)           # (N,)
        pi1 = pi1 - np.real(inter * np.conjugate(inter))
        pi2 = np.real(pi1 / r)

        total = 0.0
        i = -1
        while total < ref[j + 1]:
            i += 1
            if i >= N:
                raise RuntimeError("Sampling failed in step 2 (complex case).")
            total += pi2[i]
        echant[i] = 1

        # w <- w - (projection on e1)
        proj = w @ np.conjugate(e1)           # (N,)
        w = w - np.outer(proj, e1)            # (N, n)

        L = w[i, :]
        norm_L = np.sqrt(np.real(np.vdot(L, L)))
        if norm_L == 0:
            raise ValueError("Encountered zero-norm vector in complex DSD.")
        e1 = L / norm_L

    if B:
        return echant
    else:
        # Return 1-based indices, to match the R function
        return np.nonzero(echant == 1)[0] + 1

In [21]:
import numpy as np
from typing import Optional, Union


def spec(omega: np.ndarray, M: int, pi: Union[np.ndarray, list], spectre=100) -> np.ndarray:
    """
    Python/Numpy port of the R function:

        spec <- function(omega, M, pi, spectre = 100)

    Parameters
    ----------
    omega : array-like, shape (M, N)
        Omega matrix (same role as in the R code).
    M : int
        Integer (usually sum(pi)).
    pi : array-like, shape (N,)
        First-order inclusion probabilities.
    spectre : 100 or array-like of length M
        If 100 (default), the spectrum is constructed as in the R code.
        Otherwise, used directly as the initial spectrum.

    Returns
    -------
    mat_spectre : ndarray, shape (M, N)
    """
    pi = np.asarray(pi, dtype=float).ravel()
    N = pi.size
    omega = np.asarray(omega, dtype=float)
    if omega.shape != (M, N):
        raise ValueError("omega must have shape (M, N)")

    mat_spectre = np.zeros((M, N), dtype=float)
    pi_down = np.sort(pi)[::-1]
    mu = pi.sum()

    # --- Build initial spectrum (last column) if needed ---
    if (np.isscalar(spectre) and spectre == 100) or (
        not np.isscalar(spectre)
        and len(np.atleast_1d(spectre)) == 1
        and np.atleast_1d(spectre)[0] == 100
    ):
        spectre_vec = np.zeros(M, dtype=float)
        cumsum_1 = 0.0
        lam = 0.0

        # j in 1:(M-1)  ->  jR = j+1
        for j in range(M - 1):
            jR = j + 1
            A = max(lam, mu - cumsum_1 - (M - jR))
            B = 1.0

            # i in 1:(M-jR)
            for iR in range(1, M - jR + 1):
                # sum(pi_down[1:(M-j-i+1)]) in R
                t_idx = M - jR - iR + 1  # always >= 1 here
                s_down = pi_down[:t_idx].sum()
                B = min(B, (mu - cumsum_1 - s_down) / iR)

            B = min(B, (mu - cumsum_1) / (M - jR + 1))
            spectre_vec[j] = A + omega[j, N - 1] * (B - A)
            lam = spectre_vec[j]
            cumsum_1 += spectre_vec[j]

        # last component
        spectre_vec[M - 1] = mu - spectre_vec[: M - 1].sum()

    else:
        spectre_arr = np.asarray(spectre, dtype=float).ravel()
        if spectre_arr.size != M:
            raise ValueError("spectre must have length M")
        spectre_vec = spectre_arr

    mat_spectre[:, N - 1] = spectre_vec

    # --- Backward recursion for other columns (k in (N-1):1) ---
    for kR in range(N - 1, 0, -1):  # kR = N-1, ..., 1
        startR = max(1, M - kR + 1)
        lambda1 = mat_spectre[:, kR].copy()     # column k+1 in R
        lambda2 = mat_spectre[:, kR - 1].copy() # column k in R

        # j in start:M
        for jR in range(startR, M + 1):
            # A = max(0, lambda1[j-1], sum(lambda1[1:j]) - sum(lambda2[0:(j-1)]) - pi_down[k+1])
            if jR >= 2:
                lam_prev = lambda1[jR - 2]
            else:
                lam_prev = 0.0

            sum_l1 = lambda1[:jR].sum()
            sum_l2 = lambda2[: jR - 1].sum() if jR > 1 else 0.0
            A = max(0.0, lam_prev, sum_l1 - sum_l2 - pi_down[kR])  # pi_down[k+1] -> index kR

            # Build B_1
            B_list = []
            for iR in range(jR, M + 1):
                prem = 0.0
                deux = 0.0
                trois = lambda2[: jR - 1]

                # prem = pi_down[(M-i+1):k] if (M-i+1) <= k
                if (M - iR + 1) <= kR:
                    start_idx = M - iR + 1  # 1-based
                    prem = pi_down[start_idx - 1 : kR].sum()

                # deux = lambda1[j:(i-1)] if j <= i-1
                if jR <= (iR - 1):
                    deux = lambda1[jR - 1 : iR - 1].sum()

                B_val = prem - deux - trois.sum()
                B_list.append(B_val)

            B_inner = min(B_list) if B_list else float("inf")
            B = min(lambda1[jR - 1], B_inner)

            mat_spectre[jR - 1, kR - 1] = A + omega[jR - 1, kR - 1] * (B - A)
            lambda2[jR - 1] = mat_spectre[jR - 1, kR - 1]

    return mat_spectre


def CaDsd(
    pi: Union[np.ndarray, list],
    M: Optional[int] = None,
    omega: Optional[np.ndarray] = None,
    rho: Optional[np.ndarray] = None,
    spectre=100,
    U: Optional[np.ndarray] = None,
    option: bool = True,
):
    """
    Python/Numpy port of the R function:

        CaDsd <- function(omega=matrix(0.5,M,length(pi)),
                          rho=matrix(0.5,M,length(pi)-1),
                          M=round(sum(pi),7),
                          pi,
                          spectre=100,
                          U=diag(M),
                          option=TRUE)

    Parameters
    ----------
    pi : array-like, length N
    M : int or None
        If None, set to round(sum(pi), 7) as in R.
    omega : array-like, shape (M, N), default 0.5
    rho : array-like, shape (M, N-1), default 0.5
    spectre : passed to spec(...)
    U : initial unitary matrix, shape (M, M); default identity.
    option : kept for signature compatibility (not used here).

    Returns
    -------
    dict with keys: "K", "spectrum", "EigenBasis"
    """
    pi = np.asarray(pi, dtype=float).ravel()
    N = pi.size

    # M default
    if M is None:
        M = int(round(pi.sum(), 7))
    if int(M) != M:
        raise ValueError("M should be an integer")
    M = int(M)

    # Defaults for omega and rho
    if omega is None:
        omega = 0.5 * np.ones((M, N), dtype=float)
    else:
        omega = np.asarray(omega, dtype=float)
        if omega.shape != (M, N):
            raise ValueError("omega must have shape (M, N)")

    if rho is None:
        rho = 0.5 * np.ones((M, N - 1), dtype=float)
    else:
        rho = np.asarray(rho, dtype=float)
        if rho.shape != (M, N - 1):
            raise ValueError("rho must have shape (M, N-1)")

    # Convert rho to angles and compute spectrum
    rho = np.round(rho * 2 * np.pi, 7)
    mat_spectre = np.round(spec(omega, M, pi, spectre), 7)

    pi_down = np.sort(pi)[::-1]

    # U default
    if U is None:
        U = np.eye(M, dtype=complex)
    else:
        U = np.asarray(U, dtype=complex)
        if U.shape != (M, M):
            raise ValueError("U must have shape (M, M)")

    # phi = sqrt(pi_down[1])*U[,1] (1st column, 1-based)
    phi = np.round(np.sqrt(pi_down[0]) * U[:, [0]], 7)  # shape (M, 1)
    ens = np.arange(1, M + 1, dtype=int)               # 1..M

    # Main loop over k = 2..N
    for kR in range(2, N + 1):
        # V = diag(complex(argument = rho[,k-1], modulus = 1))
        rho_col = rho[:, kR - 2]
        V = np.diag(np.exp(1j * rho_col))

        lambda1 = mat_spectre[:, kR - 1].copy()
        lambda2 = mat_spectre[:, kR - 2].copy()

        E1 = ens.copy().tolist()
        E2 = ens.copy().tolist()

        # Matching equal eigenvalues between lambda2 and lambda1
        for jR in ens:
            val = lambda2[jR - 1]
            if not E1:
                break
            lam1_E1 = lambda1[np.array(E1) - 1]
            matches = np.where(lam1_E1 == val)[0]
            if matches.size > 0:
                # remove jR from E2
                E2 = [x for x in E2 if x != jR]
                # remove the first match from E1
                del E1[matches[0]]

        E1_arr = np.array(E1, dtype=int)
        E2_arr = np.array(E2, dtype=int)
        E1_ = M + 1 - E1_arr
        E2_ = M + 1 - E2_arr
        r = len(E1_)

        # Permutation matrices sigma1, sigma2
        if r != M:
            E1_c = ens[~np.isin(ens, E1_)]
            E2_c = ens[~np.isin(ens, E2_)]

            E1_ = np.concatenate([np.sort(E1_), np.sort(E1_c)])
            E2_ = np.concatenate([np.sort(E2_), np.sort(E2_c)])

            sigma1 = np.eye(M, dtype=complex)[E1_ - 1, :]
            sigma2 = np.eye(M, dtype=complex)[E2_ - 1, :]
        else:
            sigma1 = np.eye(M, dtype=complex)
            sigma2 = np.eye(M, dtype=complex)

        if r != 0:
            # R = diag(r)[r:1,] %*% cbind(lambda2[E2], lambda1[E1])
            lambda2_E2 = lambda2[E2_arr - 1]
            lambda1_E1 = lambda1[E1_arr - 1]
            R = np.column_stack([lambda2_E2, lambda1_E1]).astype(complex)
            R = R[::-1, :]  # reverse rows

            v = np.zeros(r, dtype=complex)
            w = np.zeros(r, dtype=complex)

            for i in range(r):
                # v-part
                v1 = R[i, 0] - R[:, 1]
                v2 = R[i, 0] - R[:, 0]
                v2[i] = 1.0 + 0j
                order = np.argsort(np.abs(v1))
                v1s = v1[order]
                v2s = v2[order]
                ratio_prod = np.prod(v1s / v2s)
                v[i] = np.round(np.sqrt(-ratio_prod), 7)

                # w-part
                w1 = R[i, 1] - R[:, 0]
                w2 = R[i, 1] - R[:, 1]
                w2[i] = 1.0 + 0j
                order_w = np.argsort(np.abs(w1))
                w1s = w1[order_w]
                w2s = w2[order_w]
                ratio_prod_w = np.prod(w1s / w2s)
                w[i] = np.round(np.sqrt(ratio_prod_w), 7)

            # vect[1:r] = v  (1-based)
            vect = np.zeros((M, 1), dtype=complex)
            vect[:r, 0] = v

            # W = (1/(t(matrix(R[,2],r,r)) - matrix(R[,1],r,r))) * (v %*% t(w))
            col_vals = R[:, 1]
            row_vals = R[:, 0]
            denom = col_vals[np.newaxis, :] - row_vals[:, np.newaxis]  # shape (r,r)
            W = (v[:, None] * w[None, :]) / denom

            # phi <- cbind(phi, U %*% V %*% t(sigma2) %*% vect)
            temp = U @ V @ sigma2.T @ vect  # (M,1)
            phi = np.hstack([phi, temp])

            # U <- U %*% V %*% t(sigma2) %*% rbind(mat1,mat2) %*% sigma1
            mat1 = np.hstack([W, np.zeros((r, M - r), dtype=complex)])
            mat2 = np.hstack([np.zeros((M - r, r), dtype=complex),
                              np.eye(M - r, dtype=complex)])
            block = np.vstack([mat1, mat2])
            U = U @ V @ sigma2.T @ block @ sigma1

    # EigenBasis <- t(Conj(phi)) %*% U %*% solve(sqrt(diag(mat_spectre[,N])))
    d = np.sqrt(mat_spectre[:, N - 1])
    if np.any(d == 0):
        raise ValueError("Zero on diagonal of spectrum, cannot invert sqrt.")
    inv_sqrt_diag = np.diag(1.0 / d).astype(complex)

    EigenBasis = phi.conj().T @ U @ inv_sqrt_diag

    # K = t(Conj(phi)) %*% phi
    K = np.round(phi.conj().T @ phi, 7)

    return {
        "K": K,                   # Gram matrix
        "spectrum": mat_spectre,  # spectrum matrix
        "EigenBasis": EigenBasis  # eigenbasis
    }

In [22]:
import numpy as np

# --------------------------------------------------------
# 1) inclusionprobabilities() – Python version
#    (size measure p -> inclusion probs π with sum π = n)
# --------------------------------------------------------

def inclusionprobabilities(p, n):
    """
    Rough equivalent of sampling::inclusionprobabilities(p, n) in R.

    p : array-like, strictly positive size measures
    n : desired fixed sample size (integer)

    Returns
    -------
    pik : ndarray of length N
        First-order inclusion probabilities, 0 < pik_i <= 1, sum pik_i = n.
    """
    p = np.asarray(p, dtype=float)
    N = p.size
    if n <= 0 or n > N:
        raise ValueError("n must be in 1..N")

    pik = np.zeros(N, dtype=float)
    mask_fixed = np.zeros(N, dtype=bool)  # True where pik is already fixed to 1
    n_rem = float(n)
    p_work = p.copy()

    # Iterative rescaling: set any prob >= 1 to 1, rescale remaining
    while True:
        idx = ~mask_fixed
        if not np.any(idx):
            break

        p_sub = p_work[idx]
        # if all remaining p_sub are zero but n_rem > 0, it's impossible
        if p_sub.sum() <= 0 and n_rem > 1e-12:
            raise RuntimeError("Cannot construct inclusion probabilities with given p and n.")

        temp = n_rem * p_sub / p_sub.sum()
        over = temp >= (1.0 - 1e-12)  # tolerance

        # If no temp >= 1, we're done
        if not np.any(over):
            pik[idx] = temp
            break

        # Fix those >=1 to exactly 1
        idx_global = np.where(idx)[0]
        over_global = idx_global[over]

        pik[over_global] = 1.0
        mask_fixed[over_global] = True
        n_rem -= over.sum()
        p_work[over_global] = 0.0

        if n_rem <= 1e-12:
            # All remaining inclusion probabilities must be zero
            break

    return pik




In [23]:
def to_r_vector(arr, name="vec"):
    arr = np.asarray(arr).ravel()
    values = ", ".join([f"{x:.10f}" for x in arr])
    return f"{name} <- c({values})"

def to_r_matrix_complex(mat, name="mat"):
    mat = np.asarray(mat)
    N, M = mat.shape
    
    r_code = f"{name} <- matrix(c(\n"
    elements = []
    
    for j in range(M):  # R fills by column
        for i in range(N):
            val = mat[i, j]
            real_part = np.real(val)
            imag_part = np.imag(val)
            
            if abs(imag_part) < 1e-10:
                elements.append(f"  {real_part:.10f}")
            else:
                sign = "+" if imag_part >= 0 else ""
                elements.append(f"  complex(real={real_part:.10f}, imaginary={imag_part:.10f})")
    
    r_code += ",\n".join(elements)
    r_code += f"\n), nrow={N}, ncol={M}, byrow=FALSE)"
    return r_code

def to_r_matrix_real(mat, name="mat"):
    mat = np.asarray(mat)
    N, M = mat.shape
    
    r_code = f"{name} <- matrix(c(\n"
    elements = []
    
    for j in range(M):  # R fills by column
        for i in range(N):
            elements.append(f"  {mat[i, j]:.10f}")
    
    r_code += ",\n".join(elements)
    r_code += f"\n), nrow={N}, ncol={M}, byrow=FALSE)"
    return r_code


# Sensitivity Analysis

In [24]:
import numpy as np
import pandas as pd

def sensitivity_analysis_uniform(
    best_omega, 
    best_rho, 
    pik_sorted,
    y_sorted, 
    z_sorted,
    var_srs_y,
    var_srs_z,
    best_var_y,
    best_var_z,
    epsilon=0.01, 
    num_perturbations=100,  
    case_name="EQUAL",
    show_matrices=True  # NEW: Show matrix details
):
 
    M, N = best_omega.shape
    I_N = np.eye(N)
    Dpi_inv = np.diag(1.0 / pik_sorted)
    n = int(round(pik_sorted.sum()))
    
    results = []
    
    print(f"\n{'='*80}")
    print(f" DEBUG SENSITIVITY ANALYSIS: {case_name} (epsilon={epsilon})")
    print(f"{'='*80}")
    print(f"\n   Configuration:")
    print(f"     N = {N}, n = {n}, M = {M}")
    print(f"     Best var_y: {best_var_y:.6f}")
    print(f"     Best var_z: {best_var_z:.6f}")
    print(f"     pik range: [{pik_sorted.min():.4f}, {pik_sorted.max():.4f}]")
    print(f"     pik sum: {pik_sorted.sum():.6f}")
    
    if show_matrices:
        print(f"\n   Best omega stats:")
        print(f"     Shape: {best_omega.shape}")
        print(f"     Range: [{best_omega.min():.4f}, {best_omega.max():.4f}]")
        print(f"     Mean: {best_omega.mean():.4f}")
        print(f"\n   Best rho stats:")
        print(f"     Shape: {best_rho.shape}")
        print(f"     Range: [{best_rho.min():.4f}, {best_rho.max():.4f}]")
        print(f"     Mean: {best_rho.mean():.4f}")
    
    # Perturbation cases
    perturbation_cases = [
        ("omega+, rho+", +epsilon, +epsilon),
        ("omega+, rho-", +epsilon, -epsilon),
        ("omega-, rho+", -epsilon, +epsilon),
        ("omega-, rho-", -epsilon, -epsilon),
        ("omega+, rho=0", +epsilon, 0),
        ("omega-, rho=0", -epsilon, 0),
        ("omega=0, rho+", 0, +epsilon),
        ("omega=0, rho-", 0, -epsilon),
    ]
    
    print(f"\n   Testing {len(perturbation_cases)} perturbation cases...")
    print(f"\n{'='*80}")
    
    for idx, (case_name_detail, delta_omega, delta_rho) in enumerate(perturbation_cases):
        print(f"\n[{idx+1}/{len(perturbation_cases)}] {case_name_detail}")
        print("-"*80)
        
        # Apply uniform perturbation
        omega_new = best_omega + delta_omega
        rho_new = best_rho + delta_rho
        
        # Clip to [0, 1]
        omega_new = np.clip(omega_new, 0.0, 1.0)
        rho_new = np.clip(rho_new, 0.0, 1.0)
        
        print(f"  Perturbation: Δω={delta_omega:+.3f}, Δρ={delta_rho:+.3f}")
        print(f"  New omega: [{omega_new.min():.4f}, {omega_new.max():.4f}]")
        print(f"  New rho:   [{rho_new.min():.4f}, {rho_new.max():.4f}]")
        
        try:
            # Build new K
            print(f"  → Calling CaDsd...")
            K_dict = CaDsd(pi=pik_sorted, M=M, omega=omega_new, rho=rho_new)
            Kmat = K_dict["K"].astype(np.complex128)
            print(f"  ✓ CaDsd succeeded")
            
            # Show K matrix stats
            if show_matrices:
                print(f"\n  K Matrix Properties:")
                print(f"    Shape: {Kmat.shape}")
                print(f"    Dtype: {Kmat.dtype}")
                print(f"    Real part range: [{np.real(Kmat).min():.4f}, {np.real(Kmat).max():.4f}]")
                print(f"    Imag part range: [{np.imag(Kmat).min():.4f}, {np.imag(Kmat).max():.4f}]")
                print(f"    Max imaginary: {np.abs(np.imag(Kmat)).max():.6f}")
            
            # CHECK 1: Diagonal validation
            print(f"\n  CHECK 1: Diagonal validation")
            diag_K = np.real(np.diag(Kmat))
            pik_sorted_desc = np.sort(pik_sorted)[::-1]  # Descending order
            
            print(f"    diag(K) range: [{diag_K.min():.6f}, {diag_K.max():.6f}]")
            print(f"    pik_sorted↓ range: [{pik_sorted_desc.min():.6f}, {pik_sorted_desc.max():.6f}]")
            
            diff_diag = np.abs(diag_K - pik_sorted_desc)
            max_diff_diag = diff_diag.max()
            print(f"    Max difference: {max_diff_diag:.6e}")
            
            if not np.allclose(diag_K, pik_sorted_desc, atol=1e-4):
                print(f"  ✗ FAILED: Diagonal mismatch (max diff = {max_diff_diag:.6e} > 1e-4)")
                
                if show_matrices:
                    print(f"\n    First 5 diagonal elements vs pik:")
                    for i in range(min(5, len(diag_K))):
                        print(f"      [{i}] diag(K)={diag_K[i]:.6f}, pik={pik_sorted_desc[i]:.6f}, diff={diff_diag[i]:.6e}")
                
                results.append({
                    'perturbation_type': case_name_detail,
                    'delta_omega': delta_omega,
                    'delta_rho': delta_rho,
                    'original_var_y': best_var_y,
                    'original_var_z': best_var_z,
                    'var_y': np.nan,
                    'var_z': np.nan,
                    'delta_var_y': np.nan,
                    'delta_var_z': np.nan,
                    'valid': False,
                    'reason': f'diagonal mismatch (max={max_diff_diag:.6e})'
                })
                continue
            
            print(f"  ✓ PASSED: Diagonal OK")
            
            # CHECK 2: Eigenvalues
            print(f"\n  CHECK 2: Eigenvalue validation")
            evals = np.linalg.eigvalsh(Kmat)
            
            print(f"    Eigenvalues range: [{evals.min():.6f}, {evals.max():.6f}]")
            print(f"    Eigenvalues sum: {evals.sum():.6f} (should be ≈ {n})")
            
            num_negative = np.sum(evals < -1e-4)
            num_above_one = np.sum(evals > 1 + 1e-4)
            
            print(f"    Negative (< -1e-4): {num_negative}")
            print(f"    Above 1 (> 1+1e-4): {num_above_one}")
            
            if not (np.all(evals >= -1e-4) and np.all(evals <= 1 + 1e-4)):
                print(f"  ✗ FAILED: Eigenvalues out of [0,1]")
                
                if show_matrices:
                    print(f"\n    Problematic eigenvalues:")
                    bad_evals = evals[(evals < -1e-4) | (evals > 1 + 1e-4)]
                    for i, ev in enumerate(bad_evals[:5]):  # Show first 5
                        print(f"      {ev:.6f}")
                
                results.append({
                    'perturbation_type': case_name_detail,
                    'delta_omega': delta_omega,
                    'delta_rho': delta_rho,
                    'original_var_y': best_var_y,
                    'original_var_z': best_var_z,
                    'var_y': np.nan,
                    'var_z': np.nan,
                    'delta_var_y': np.nan,
                    'delta_var_z': np.nan,
                    'valid': False,
                    'reason': f'eigenvalues: {num_negative} negative, {num_above_one} > 1'
                })
                continue
            
            print(f"  ✓ PASSED: Eigenvalues in [0,1]")
            
            # CHECK 3: Trace
            print(f"\n  CHECK 3: Trace validation")
            trace_K = evals.sum()
            trace_diff = abs(trace_K - n)
            
            print(f"    Trace(K) = {trace_K:.6f}")
            print(f"    Target n = {n}")
            print(f"    Difference = {trace_diff:.6e}")
            
            if not np.isclose(trace_K, n, atol=1e-4):
                print(f"  ✗ FAILED: Trace ≠ n (diff = {trace_diff:.6e})")
                
                results.append({
                    'perturbation_type': case_name_detail,
                    'delta_omega': delta_omega,
                    'delta_rho': delta_rho,
                    'original_var_y': best_var_y,
                    'original_var_z': best_var_z,
                    'var_y': np.nan,
                    'var_z': np.nan,
                    'delta_var_y': np.nan,
                    'delta_var_z': np.nan,
                    'valid': False,
                    'reason': f'trace != n (diff={trace_diff:.6e})'
                })
                continue
            
            print(f"  ✓ PASSED: Trace = n")
            
            # Calculate variances
            print(f"\n  Computing variances...")
            A = Kmat * (I_N - Kmat.conj())
            var_y_new = float(np.real(y_sorted.conj().T @ Dpi_inv @ A @ Dpi_inv @ y_sorted))
            var_z_new = float(np.real(z_sorted.conj().T @ Dpi_inv @ A @ Dpi_inv @ z_sorted))
            
            delta_var_y = abs(var_y_new - best_var_y)
            delta_var_z = abs(var_z_new - best_var_z)
            
            print(f"    var_y: {best_var_y:.6f} → {var_y_new:.6f} (Δ = {delta_var_y:.6f})")
            print(f"    var_z: {best_var_z:.6f} → {var_z_new:.6f} (Δ = {delta_var_z:.6f})")
            
            print(f"\n  ✓✓✓ VALID DESIGN! ✓✓✓")
            
            results.append({
                'perturbation_type': case_name_detail,
                'delta_omega': delta_omega,
                'delta_rho': delta_rho,
                'var_y': var_y_new,
                'var_z': var_z_new,
                'original_var_y': best_var_y,
                'original_var_z': best_var_z,
                'delta_var_y': delta_var_y,
                'delta_var_z': delta_var_z,
                'valid': True,
                'reason': 'OK'
            })
            
        except Exception as e:
            print(f"\n  ✗✗✗ EXCEPTION: {str(e)}")
            import traceback
            if show_matrices:
                print(f"\n  Traceback:")
                traceback.print_exc()
            
            results.append({
                'perturbation_type': case_name_detail,
                'delta_omega': delta_omega,
                'delta_rho': delta_rho,
                'original_var_y': best_var_y,
                'original_var_z': best_var_z,
                'var_y': np.nan,
                'var_z': np.nan,
                'delta_var_y': np.nan,
                'delta_var_z': np.nan,
                'valid': False,
                'reason': f'Exception: {str(e)[:50]}'
            })
    
    # Summary
    df = pd.DataFrame(results)
    num_valid = len(df[df['valid'] == True])
    num_total = len(df)
    
    print(f"\n{'='*80}")
    print(f"SUMMARY")
    print(f"{'='*80}")
    print(f"  Valid designs: {num_valid}/{num_total} ({100*num_valid/num_total:.1f}%)")
    
    if num_valid > 0:
        df_valid = df[df['valid'] == True]
        print(f"\n  Valid cases:")
        for _, row in df_valid.iterrows():
            print(f"    • {row['perturbation_type']}: Δvar_y={row['delta_var_y']:.6f}, Δvar_z={row['delta_var_z']:.6f}")
    else:
        print(f"\n   NO VALID DESIGNS FOUND!")
        print(f"\n  Failure reasons:")
        for reason in df['reason'].value_counts().items():
            print(f"    • {reason[0]}: {reason[1]} cases")
    
    print(f"\n{'='*80}\n")
    
    return df


print("DEBUG sensitivity analysis function loaded!")


DEBUG sensitivity analysis function loaded!


In [25]:
# Uniform sensitivity analysis for EQUAL case (epsilon = 0.01)
if best_K_equal is not None and best_omega_equal is not None:
    # Calculate best variances
    Dpi_inv_eq = np.diag(1.0 / pik_equal_sorted)
    A_best_eq = best_K_equal * (I_N - best_K_equal.conj())
    best_var_y_eq = float(np.real(y_sorted.conj().T @ Dpi_inv_eq @ A_best_eq @ Dpi_inv_eq @ y_sorted))
    best_var_z_eq = float(np.real(z_sorted.conj().T @ Dpi_inv_eq @ A_best_eq @ Dpi_inv_eq @ z_sorted))
    
    df_uniform_equal_001 = sensitivity_analysis_uniform(
        best_omega=best_omega_equal,
        best_rho=best_rho_equal,
        pik_sorted=pik_equal_sorted,
        y_sorted=y_sorted,
        z_sorted=z_sorted,
        var_srs_y=var_srs_y,
        var_srs_z=var_srs_z,
        best_var_y=best_var_y_eq,
        best_var_z=best_var_z_eq,
        epsilon=0.01,
        case_name="EQUAL"
    )
else:
    print("  No best design found for EQUAL case!")
    

NameError: name 'best_K_equal' is not defined

In [ ]:
# Uniform sensitivity analysis for EQUAL case (epsilon = 0.02)
if best_K_equal is not None and best_omega_equal is not None:
    df_uniform_equal_002 = sensitivity_analysis_uniform(
        best_omega=best_omega_equal,
        best_rho=best_rho_equal,
        pik_sorted=pik_equal_sorted,
        y_sorted=y_sorted,
        z_sorted=z_sorted,
        var_srs_y=var_srs_y,
        var_srs_z=var_srs_z,
        best_var_y=best_var_y_eq,
        best_var_z=best_var_z_eq,
        epsilon=0.02,
        case_name="EQUAL"
    )
    


 DEBUG SENSITIVITY ANALYSIS: EQUAL (epsilon=0.02)

   Configuration:
     N = 25, n = 5, M = 5
     Best var_y: 2307.139851
     Best var_z: 2494.872087
     pik range: [0.2000, 0.2000]
     pik sum: 5.000000

   Best omega stats:
     Shape: (5, 25)
     Range: [0.0008, 0.9997]
     Mean: 0.4988

   Best rho stats:
     Shape: (5, 24)
     Range: [0.0009, 0.9739]
     Mean: 0.4774

   Testing 8 perturbation cases...


[1/8] omega+, rho+
--------------------------------------------------------------------------------
  Perturbation: Δω=+0.020, Δρ=+0.020
  New omega: [0.0208, 1.0000]
  New rho:   [0.0209, 0.9939]
  → Calling CaDsd...
  ✓ CaDsd succeeded

  K Matrix Properties:
    Shape: (25, 25)
    Dtype: complex128
    Real part range: [-0.1708, 0.2000]
    Imag part range: [-0.1886, 0.1886]
    Max imaginary: 0.188570

  CHECK 1: Diagonal validation
    diag(K) range: [0.199997, 0.200000]
    pik_sorted↓ range: [0.200000, 0.200000]
    Max difference: 2.700000e-06
  ✓ PASSED: Diago

In [ ]:
# Uniform sensitivity analysis for UNEQUAL case (epsilon = 0.01)
if best_K_unequal is not None and best_omega_unequal is not None:
    # Calculate best variances
    Dpi_inv_uneq = np.diag(1.0 / pik_unequal_sorted)
    A_best_uneq = best_K_unequal * (I_N - best_K_unequal.conj())
    best_var_y_uneq = float(np.real(y_sorted_uneq.conj().T @ Dpi_inv_uneq @ A_best_uneq @ Dpi_inv_uneq @ y_sorted_uneq))
    best_var_z_uneq = float(np.real(z_sorted_uneq.conj().T @ Dpi_inv_uneq @ A_best_uneq @ Dpi_inv_uneq @ z_sorted_uneq))
    
    df_uniform_unequal_001 = sensitivity_analysis_uniform(
        best_omega=best_omega_unequal,
        best_rho=best_rho_unequal,
        pik_sorted=pik_unequal_sorted,
        y_sorted=y_sorted_uneq,
        z_sorted=z_sorted_uneq,
        var_srs_y=var_srs_y,
        var_srs_z=var_srs_z,
        best_var_y=best_var_y_uneq,
        best_var_z=best_var_z_uneq,
        epsilon=0.01,
        case_name="UNEQUAL"
    )
else:
    print("  No best design found for UNEQUAL case!")


 DEBUG SENSITIVITY ANALYSIS: UNEQUAL (epsilon=0.01)

   Configuration:
     N = 25, n = 5, M = 5
     Best var_y: 872.279234
     Best var_z: 1011.023084
     pik range: [0.1724, 0.2269]
     pik sum: 5.000000

   Best omega stats:
     Shape: (5, 25)
     Range: [0.0024, 0.9886]
     Mean: 0.4743

   Best rho stats:
     Shape: (5, 24)
     Range: [0.0237, 0.9991]
     Mean: 0.5113

   Testing 8 perturbation cases...


[1/8] omega+, rho+
--------------------------------------------------------------------------------
  Perturbation: Δω=+0.010, Δρ=+0.010
  New omega: [0.0124, 0.9986]
  New rho:   [0.0337, 1.0000]
  → Calling CaDsd...
  ✓ CaDsd succeeded

  K Matrix Properties:
    Shape: (25, 25)
    Dtype: complex128
    Real part range: [-0.1574, 0.2269]
    Imag part range: [-0.1742, 0.1742]
    Max imaginary: 0.174250

  CHECK 1: Diagonal validation
    diag(K) range: [0.172451, 0.226896]
    pik_sorted↓ range: [0.172449, 0.226896]
    Max difference: 4.437068e-06
  ✓ PASSED: Diag

In [ ]:
# Uniform sensitivity analysis for UNEQUAL case (epsilon = 0.02)
if best_K_unequal is not None and best_omega_unequal is not None:
    df_uniform_unequal_002 = sensitivity_analysis_uniform(
        best_omega=best_omega_unequal,
        best_rho=best_rho_unequal,
        pik_sorted=pik_unequal_sorted,
        y_sorted=y_sorted_uneq,
        z_sorted=z_sorted_uneq,
        var_srs_y=var_srs_y,
        var_srs_z=var_srs_z,
        best_var_y=best_var_y_uneq,
        best_var_z=best_var_z_uneq,
        epsilon=0.02,
        case_name="UNEQUAL"
    )
    


 DEBUG SENSITIVITY ANALYSIS: UNEQUAL (epsilon=0.02)

   Configuration:
     N = 25, n = 5, M = 5
     Best var_y: 872.279234
     Best var_z: 1011.023084
     pik range: [0.1724, 0.2269]
     pik sum: 5.000000

   Best omega stats:
     Shape: (5, 25)
     Range: [0.0024, 0.9886]
     Mean: 0.4743

   Best rho stats:
     Shape: (5, 24)
     Range: [0.0237, 0.9991]
     Mean: 0.5113

   Testing 8 perturbation cases...


[1/8] omega+, rho+
--------------------------------------------------------------------------------
  Perturbation: Δω=+0.020, Δρ=+0.020
  New omega: [0.0224, 1.0000]
  New rho:   [0.0437, 1.0000]
  → Calling CaDsd...
  ✓ CaDsd succeeded

  K Matrix Properties:
    Shape: (25, 25)
    Dtype: complex128
    Real part range: [-0.1667, 0.2269]
    Imag part range: [-0.1664, 0.1664]
    Max imaginary: 0.166424

  CHECK 1: Diagonal validation
    diag(K) range: [0.172456, 0.226896]
    pik_sorted↓ range: [0.172449, 0.226896]
    Max difference: 1.666293e-05
  ✓ PASSED: Diag

In [ ]:
print("\n" + "="*80)
print(" UNIFORM PERTURBATION RESULTS SUMMARY")
print("="*80)

# Combine all results
all_results = []

# EQUAL - epsilon 0.01
if 'df_uniform_equal_001' in locals():
    df_temp = df_uniform_equal_001.copy()
    df_temp['case'] = 'EQUAL'
    df_temp['epsilon'] = 0.01
    all_results.append(df_temp)

# EQUAL - epsilon 0.02
if 'df_uniform_equal_002' in locals():
    df_temp = df_uniform_equal_002.copy()
    df_temp['case'] = 'EQUAL'
    df_temp['epsilon'] = 0.02
    all_results.append(df_temp)

# UNEQUAL - epsilon 0.01
if 'df_uniform_unequal_001' in locals():
    df_temp = df_uniform_unequal_001.copy()
    df_temp['case'] = 'UNEQUAL'
    df_temp['epsilon'] = 0.01
    all_results.append(df_temp)

# UNEQUAL - epsilon 0.02
if 'df_uniform_unequal_002' in locals():
    df_temp = df_uniform_unequal_002.copy()
    df_temp['case'] = 'UNEQUAL'
    df_temp['epsilon'] = 0.02
    all_results.append(df_temp)

if all_results:
    df_all_uniform = pd.concat(all_results, ignore_index=True)
    
    if 'original_var_y' in df_all_uniform.columns and 'original_var_z' in df_all_uniform.columns:
        df_all_uniform['percentage_change_var_y'] = ((df_all_uniform['delta_var_y'] / df_all_uniform['original_var_y']) * 100).round(2)
        df_all_uniform['percentage_change_var_z'] = ((df_all_uniform['delta_var_z'] / df_all_uniform['original_var_z']) * 100).round(2)
    else:
        print(" Warning: original_var_y and original_var_z columns not found!")
        print("   Please add them during perturbation testing.")
    
    # نمایش جدول
    print("\n Complete Results Table:")
    columns_to_show = ['case', 'epsilon', 'perturbation_type', 
                       'percentage_change_var_y', 'percentage_change_var_z',
                       'valid', 'reason']
    
    existing_columns = [col for col in columns_to_show if col in df_all_uniform.columns]
    print(df_all_uniform[existing_columns].to_string(index=False))
    
    # Summary statistics
    print("\n" + "="*80)
    print(" SUMMARY BY CASE AND EPSILON")
    print("="*80)
    
    for case in ['EQUAL', 'UNEQUAL']:
        for eps in [0.01, 0.02]:
            df_subset = df_all_uniform[(df_all_uniform['case'] == case) & 
                                       (df_all_uniform['epsilon'] == eps)]
            df_valid = df_subset[df_subset['valid'] == True]
            
            print(f"\n{case}, ε={eps}:")
            print(f"   Valid: {len(df_valid)}/{len(df_subset)} ({100*len(df_valid)/len(df_subset):.1f}%)")
            
            if len(df_valid) > 0:
                if 'percentage_change_var_y' in df_valid.columns:
                    print(f"   % Change var_y: mean={df_valid['percentage_change_var_y'].mean():.2f}%, "
                          f"min={df_valid['percentage_change_var_y'].min():.2f}%, "
                          f"max={df_valid['percentage_change_var_y'].max():.2f}%")
                
                if 'percentage_change_var_z' in df_valid.columns:
                    print(f"   % Change var_z: mean={df_valid['percentage_change_var_z'].mean():.2f}%, "
                          f"min={df_valid['percentage_change_var_z'].min():.2f}%, "
                          f"max={df_valid['percentage_change_var_z'].max():.2f}%")
else:
    print("  No results available!")

print("\n" + "="*80)
print(" Uniform sensitivity analysis complete!")
print("="*80)


 UNIFORM PERTURBATION RESULTS SUMMARY

 Complete Results Table:
   case  epsilon perturbation_type  percentage_change_var_y  percentage_change_var_z  valid                         reason
  EQUAL     0.01      omega+, rho+                     3.15                     5.86   True                             OK
  EQUAL     0.01      omega+, rho-                     3.15                     5.86   True                             OK
  EQUAL     0.01      omega-, rho+                     3.28                     5.28   True                             OK
  EQUAL     0.01      omega-, rho-                     3.28                     5.28   True                             OK
  EQUAL     0.01     omega+, rho=0                     3.15                     5.86   True                             OK
  EQUAL     0.01     omega-, rho=0                     3.28                     5.28   True                             OK
  EQUAL     0.01     omega=0, rho+                     2.66               

# ABC Algorithm

In [28]:
import numpy as np
import time
from typing import Tuple, List, Optional

class ABCAlgorithm:
    def __init__(self, y_sorted, z_sorted, pik_sorted, var_srs_y, var_srs_z, M, n, case_name=""):
        self.y_sorted = y_sorted
        self.z_sorted = z_sorted
        self.pik_sorted = pik_sorted
        self.var_srs_y = var_srs_y
        self.var_srs_z = var_srs_z
        self.M = M
        self.n = n
        self.N = len(pik_sorted)
        self.case_name = case_name
        
        # Setup matrices
        self.I_N = np.eye(self.N)
        self.Dpi_inv = np.diag(1.0 / pik_sorted)
        
        # IMPORTANT: CaDsd expects pik in DESCENDING order for validation
        self.pik_sorted_desc = np.sort(pik_sorted)[::-1]
        
        # Calculate P_π optimal
        self._calculate_optimal()
        
        # Best solution tracking
        self.global_best_eff_z = 0.0
        self.global_best_eff_y = 0.0
        self.global_best_omega = None
        self.global_best_rho = None
        
        # History
        self.history = []
        self.scout_history = []
        
    def _calculate_optimal(self):
        Base_opt = Ppi(self.pik_sorted)
        Ppi_mat = Base_opt @ Base_opt.T
        A_opt = (self.I_N - Ppi_mat) * Ppi_mat
        
        var_opt_y = float(self.y_sorted.T @ self.Dpi_inv @ A_opt @ self.Dpi_inv @ self.y_sorted)
        var_opt_z = float(self.z_sorted.T @ self.Dpi_inv @ A_opt @ self.Dpi_inv @ self.z_sorted)
        
        self.eff_y_optimal = self.var_srs_y / var_opt_y
        self.eff_z_optimal = self.var_srs_z / var_opt_z
    
    def evaluate(self, omega: np.ndarray, rho: np.ndarray) -> Tuple[float, float, bool]:
        """
        Evaluate a solution (omega, rho)
        
        FIXED: Now uses pik_sorted_desc for proper validation
        
        Returns:
        --------
        (eff_z, eff_y, valid) : tuple
        """
        try:
            K_dict = CaDsd(pi=self.pik_sorted, M=self.M, omega=omega, rho=rho)
            Kmat = K_dict["K"].astype(np.complex128)
        except:
            return (0.0, 0.0, False)
        
        # FIXED: Validation with DESCENDING sorted pik
        diag_K = np.real(np.diag(Kmat))
        
        # CaDsd produces diagonal in DESCENDING order
        if not np.allclose(diag_K, self.pik_sorted_desc, atol=1e-3):
            return (0.0, 0.0, False)
        
        evals = np.linalg.eigvalsh(Kmat)
        if not (np.all(evals >= -1e-4) and np.all(evals <= 1 + 1e-4)):
            return (0.0, 0.0, False)
        if not np.isclose(evals.sum(), self.n, atol=1e-4):
            return (0.0, 0.0, False)
        
        # Calculate efficiency
        A = Kmat * (self.I_N - Kmat.conj())
        var_y = float(np.real(self.y_sorted.conj().T @ self.Dpi_inv @ A @ self.Dpi_inv @ self.y_sorted))
        var_z = float(np.real(self.z_sorted.conj().T @ self.Dpi_inv @ A @ self.Dpi_inv @ self.z_sorted))
        
        eff_y = self.var_srs_y / var_y
        eff_z = self.var_srs_z / var_z
        
        return (eff_z, eff_y, True)
    
    def initialize_population(self, colony_size: int, verbose: bool = True) -> List[dict]:
        """Initialize food sources (population)"""
        if verbose:
            print(f" Initializing {colony_size} food sources...")
        
        population = []
        count = 0
        max_attempts = colony_size * 20  # Increase attempts
        
        while len(population) < colony_size and count < max_attempts:
            omega = np.random.rand(self.M, self.N)
            rho = np.random.rand(self.M, self.N - 1)
            
            eff_z, eff_y, valid = self.evaluate(omega, rho)
            
            if valid:
                population.append({
                    'omega': omega,
                    'rho': rho,
                    'eff_z': eff_z,
                    'eff_y': eff_y,
                    'trial': 0
                })
                
                if eff_z > self.global_best_eff_z:
                    self.global_best_eff_z = eff_z
                    self.global_best_eff_y = eff_y
                    self.global_best_omega = omega
                    self.global_best_rho = rho
            
            count += 1
            if verbose and count % 100 == 0:
                print(f"   Evaluated {count}, found {len(population)}/{colony_size} valid sources", end="\r")
        
        if verbose:
            print(f"\n    Initialized {len(population)} food sources (out of {colony_size} requested)")
            if len(population) > 0:
                print(f"    Best initial: eff_z={self.global_best_eff_z:.4f}")
            else:
                print(f"    ⚠️  WARNING: No valid solutions found!")
        
        return population
    
    def employed_bee_phase(self, population: List[dict]) -> List[dict]:
        """Phase 1: Employed bees search around their food sources"""
        if len(population) == 0:
            return population
            
        new_population = []
        
        for i, food in enumerate(population):
            k = np.random.choice([j for j in range(len(population)) if j != i]) if len(population) > 1 else 0
            
            phi_omega = np.random.uniform(-1, 1, food['omega'].shape)
            phi_rho = np.random.uniform(-1, 1, food['rho'].shape)
            
            new_omega = food['omega'] + phi_omega * (food['omega'] - population[k]['omega'])
            new_rho = food['rho'] + phi_rho * (food['rho'] - population[k]['rho'])
            
            new_omega = np.clip(new_omega, 0, 1)
            new_rho = np.clip(new_rho, 0, 1)
            
            eff_z, eff_y, valid = self.evaluate(new_omega, new_rho)
            
            if valid and eff_z > food['eff_z']:
                new_population.append({
                    'omega': new_omega,
                    'rho': new_rho,
                    'eff_z': eff_z,
                    'eff_y': eff_y,
                    'trial': 0
                })
                
                if eff_z > self.global_best_eff_z:
                    self.global_best_eff_z = eff_z
                    self.global_best_eff_y = eff_y
                    self.global_best_omega = new_omega
                    self.global_best_rho = new_rho
            else:
                new_population.append({
                    'omega': food['omega'],
                    'rho': food['rho'],
                    'eff_z': food['eff_z'],
                    'eff_y': food['eff_y'],
                    'trial': food['trial'] + 1
                })
        
        return new_population
    
    def onlooker_bee_phase(self, population: List[dict]) -> List[dict]:
        """Phase 2: Onlooker bees select food sources based on probability"""
        if len(population) == 0:
            return population
            
        fitness = np.array([food['eff_z'] for food in population])
        
        if fitness.sum() == 0:
            prob = np.ones(len(population)) / len(population)
        else:
            prob = fitness / fitness.sum()
        
        new_population = []
        
        for i, food in enumerate(population):
            if np.random.rand() < prob[i]:
                k = np.random.choice([j for j in range(len(population)) if j != i]) if len(population) > 1 else 0
                
                phi_omega = np.random.uniform(-1, 1, food['omega'].shape)
                phi_rho = np.random.uniform(-1, 1, food['rho'].shape)
                
                new_omega = food['omega'] + phi_omega * (food['omega'] - population[k]['omega'])
                new_rho = food['rho'] + phi_rho * (food['rho'] - population[k]['rho'])
                
                new_omega = np.clip(new_omega, 0, 1)
                new_rho = np.clip(new_rho, 0, 1)
                
                eff_z, eff_y, valid = self.evaluate(new_omega, new_rho)
                
                if valid and eff_z > food['eff_z']:
                    new_population.append({
                        'omega': new_omega,
                        'rho': new_rho,
                        'eff_z': eff_z,
                        'eff_y': eff_y,
                        'trial': 0
                    })
                    
                    if eff_z > self.global_best_eff_z:
                        self.global_best_eff_z = eff_z
                        self.global_best_eff_y = eff_y
                        self.global_best_omega = new_omega
                        self.global_best_rho = new_rho
                else:
                    new_population.append({
                        'omega': food['omega'],
                        'rho': food['rho'],
                        'eff_z': food['eff_z'],
                        'eff_y': food['eff_y'],
                        'trial': food['trial'] + 1
                    })
            else:
                new_population.append(food)
        
        return new_population
    
    def scout_bee_phase(self, population: List[dict], limit: int, verbose: bool = False) -> Tuple[List[dict], int]:
        """Phase 3: Scout bees abandon exhausted sources and find new ones"""
        if len(population) == 0:
            return population, 0
            
        new_population = []
        n_abandoned = 0
        
        for food in population:
            if food['trial'] > limit:
                n_abandoned += 1
                
                attempts = 0
                found = False
                while attempts < 50:
                    omega = np.random.rand(self.M, self.N)
                    rho = np.random.rand(self.M, self.N - 1)
                    
                    eff_z, eff_y, valid = self.evaluate(omega, rho)
                    
                    if valid:
                        new_population.append({
                            'omega': omega,
                            'rho': rho,
                            'eff_z': eff_z,
                            'eff_y': eff_y,
                            'trial': 0
                        })
                        
                        if eff_z > self.global_best_eff_z:
                            self.global_best_eff_z = eff_z
                            self.global_best_eff_y = eff_y
                            self.global_best_omega = omega
                            self.global_best_rho = rho
                        
                        found = True
                        break
                    
                    attempts += 1
                
                if not found:
                    new_population.append(food)
            else:
                new_population.append(food)
        
        return new_population, n_abandoned
    
    def optimize(self, colony_size: int = 50, max_iterations: int = 100, limit: int = 20, verbose: bool = True) -> dict:
        """Run complete ABC Algorithm"""
        start_time = time.time()
        
        if verbose:
            print("=" * 80)
            print(f" ABC ALGORITHM - {self.case_name}")
            print("=" * 80)
            print(f"\n Configuration:")
            print(f"   Colony size: {colony_size}")
            print(f"   Max iterations: {max_iterations}")
            print(f"   Abandonment limit: {limit}")
            print(f"\n P_π optimal (target):")
            print(f"   eff_z = {self.eff_z_optimal:.4f}")
            print(f"   eff_y = {self.eff_y_optimal:.4f}\n")
        
        # Initialize population
        population = self.initialize_population(colony_size, verbose)
        
        if len(population) == 0:
            if verbose:
                print("\n FAILED: Could not initialize any valid solutions!")
                print("   This indicates the problem is too constrained for random initialization.")
            
            return self._prepare_results(time.time() - start_time, colony_size, 0)
        
        # Main loop
        if verbose:
            print(f"\n ABC Optimization ({max_iterations} iterations)...\n")
        
        total_abandoned = 0
        
        for iteration in range(max_iterations):
            population = self.employed_bee_phase(population)
            population = self.onlooker_bee_phase(population)
            population, n_abandoned = self.scout_bee_phase(population, limit, verbose=False)
            total_abandoned += n_abandoned
            
            self.history.append(self.global_best_eff_z)
            self.scout_history.append(n_abandoned)
            
            if verbose and (iteration + 1) % 10 == 0:
                avg_trial = np.mean([f['trial'] for f in population]) if len(population) > 0 else 0
                gap = ((self.global_best_eff_z / self.eff_z_optimal) - 1) * 100 if self.eff_z_optimal > 0 else 0
                
                print(f"   [{iteration+1:3d}/{max_iterations}] "
                      f"best={self.global_best_eff_z:.4f}, gap={gap:+.2f}%, "
                      f"avg_trial={avg_trial:.1f}, scouts={n_abandoned}", end="\r")
        
        if verbose:
            print("\n")
        
        total_time = time.time() - start_time
        
        return self._prepare_results(total_time, colony_size, total_abandoned)
    
    def _prepare_results(self, total_time: float, colony_size: int, total_abandoned: int) -> dict:
        """Prepare results dictionary"""
        success = self.global_best_eff_z > self.eff_z_optimal
        
        if self.eff_z_optimal > 0 and self.global_best_eff_z > 0:
            gap = ((self.eff_z_optimal / self.global_best_eff_z) - 1) * 100
            improvement = ((self.global_best_eff_z / self.eff_z_optimal) - 1) * 100
        else:
            gap = float('inf')
            improvement = 0
        
        return {
            'case': self.case_name,
            'best_eff_z': self.global_best_eff_z,
            'best_eff_y': self.global_best_eff_y,
            'best_omega': self.global_best_omega,
            'best_rho': self.global_best_rho,
            'optimal_eff_z': self.eff_z_optimal,
            'optimal_eff_y': self.eff_y_optimal,
            'success': success,
            'gap_percent': gap,
            'improvement_percent': improvement if success else 0,
            'total_time': total_time,
            'colony_size': colony_size,
            'total_abandoned': total_abandoned,
            'history': self.history.copy(),
            'scout_history': self.scout_history.copy()
        }
    
    def print_results(self, results: dict):
        """Print formatted results"""
        print("=" * 80)
        print(" FINAL RESULTS (ABC)")
        print("=" * 80)
        
        print(f"\n Best solution:")
        print(f"   eff_z = {results['best_eff_z']:.6f}")
        print(f"   eff_y = {results['best_eff_y']:.6f}")
        
        print(f"\n P_π optimal:")
        print(f"   eff_z = {results['optimal_eff_z']:.6f}")
        print(f"   eff_y = {results['optimal_eff_y']:.6f}")
        
        print(f"\n Comparison:")
        if results['success']:
            print(f"    SUCCESS! {results['improvement_percent']:.2f}% better than P_π!")
        elif results['best_eff_z'] == 0:
            print(f"    FAILED: No valid solution found")
        elif results['gap_percent'] < 1:
            print(f"    Very close! Gap: {results['gap_percent']:.2f}%")
        else:
            print(f"    Gap to P_π: {results['gap_percent']:.2f}%")
        
        print(f"\n  Performance:")
        print(f"   Time: {results['total_time']:.2f}s")
        print(f"   Colony size: {results['colony_size']}")
        print(f"   Total abandoned: {results['total_abandoned']}")
        
        if len(results['history']) > 0 and results['history'][0] > 0:
            print(f"\n Convergence:")
            print(f"   Start: {results['history'][0]:.4f}")
            print(f"   End: {results['history'][-1]:.4f}")
            conv = ((results['history'][-1] / results['history'][0]) - 1) * 100
            print(f"   Improvement: {conv:.2f}%")
        
        print("=" * 80)


print("   ABCAlgorithm class loaded!")

   ABCAlgorithm class loaded!


# Run ABC on MU284

In [31]:
import pandas as pd
import numpy as np

# Read data
df = pd.read_csv('/home/bardia/projects/graphical-sampling/simulations_abc/populations/real/MU284.csv')

# Calculate correlations with P85 (y)
y = df['P85']

correlations = {
    'P75': np.corrcoef(df['P75'], y)[0, 1],
    'S82': np.corrcoef(df['S82'], y)[0, 1],
    'ME84': np.corrcoef(df['ME84'], y)[0, 1],
    'REV84': np.corrcoef(df['REV84'], y)[0, 1],
    'REG': np.corrcoef(df['REG'], y)[0, 1],
}

print("Correlations with P85 (y):")
print("=" * 40)
for col, corr in sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True):
    print(f"ρ({col}, P85) = {corr:.4f}")

print("\n" + "=" * 40)
print("Mapping to paper's notation:")
print("=" * 40)
print("For ρ(x,y) ≈ 0.995  → use x = P75")
print("For ρ(z,y) ≈ 0.988  → use z = ME84")
print("For ρ(z,y) ≈ 0.913  → use z = REV84")
print("For ρ(z,y) ≈ 0.865  → use z = S82")
print("For ρ(z,y) ≈ -0.186 → use z = REG")

print("\n" + "=" * 40)
print("Test combinations structure:")
print("=" * 40)
print("z options: ME84, REV84, REG, S82, Equal")
print("x options: P75, S82, REG, Equal")
print("N_node options: 5, 10, 20, 40")
print(f"Total records: N = {len(df)}")
print("Sample size: n = 14")


Correlations with P85 (y):
ρ(P75, P85) = 0.9950
ρ(ME84, P85) = 0.9875
ρ(REV84, P85) = 0.9132
ρ(S82, P85) = 0.8654
ρ(REG, P85) = -0.1862

Mapping to paper's notation:
For ρ(x,y) ≈ 0.995  → use x = P75
For ρ(z,y) ≈ 0.988  → use z = ME84
For ρ(z,y) ≈ 0.913  → use z = REV84
For ρ(z,y) ≈ 0.865  → use z = S82
For ρ(z,y) ≈ -0.186 → use z = REG

Test combinations structure:
z options: ME84, REV84, REG, S82, Equal
x options: P75, S82, REG, Equal
N_node options: 5, 10, 20, 40
Total records: N = 281
Sample size: n = 14


In [ ]:
import numpy as np
import pandas as pd

print("="*80)
print("ABC ON FULL MU284 DATA (N=281)")
print("="*80)

# Load FULL data
# df = pd.read_csv('MU284_filtered.csv')

# Test cases
test_cases = [
    ('ME84', 'P75', 'High Corr', 0.988),
    ('S82', 'P75', 'Medium Corr', 0.865),
    ('REG', 'P75', 'Low Corr', -0.186),
]

sample_sizes = [5, 10, 20, 40]

y_var = 'P85'
N = 281

all_results = []

for z_name, x_name, corr_label, expected_corr in test_cases:
    
    print(f"\n{'='*80}")
    print(f"{corr_label}: z = {z_name}")
    print(f"{'='*80}")
    
    y_raw = df[y_var].values
    z_raw = df[z_name].values
    x_raw = df[x_name].values
    
    actual_corr = np.corrcoef(y_raw, z_raw)[0,1]
    print(f"Corr(P85, {z_name}) = {actual_corr:.3f}")
    print(f"N = {N} (FULL dataset)")
    
    for n in sample_sizes:
        print(f"\n  n = {n}:")
        
        # Create π ∝ x
        pik_raw = inclusionprobabilities(x_raw, n)
        
        # Sort by z/π
        z_over_pi = z_raw / pik_raw
        sort_idx = np.argsort(z_over_pi)
        
        y_sorted = y_raw[sort_idx]
        z_sorted = z_raw[sort_idx]
        pik_sorted = pik_raw[sort_idx]
        
        # SRS
        var_srs_y = N**2 * (1 - n/N) * np.var(y_raw, ddof=1) / n
        var_srs_z = N**2 * (1 - n/N) * np.var(z_raw, ddof=1) / n
        
        # ABC with PROPER parameters
        # Adjusted based on n (larger n = harder problem)
        if n == 5:
            colony_size = 200
            max_iterations = 1000
            limit = 100
        elif n == 10:
            colony_size = 150
            max_iterations = 800
            limit = 80
        elif n == 20:
            colony_size = 100
            max_iterations = 500
            limit = 50
        else:  # n == 40
            colony_size = 100
            max_iterations = 300
            limit = 50
        
        print(f"    ABC params: colony={colony_size}, iter={max_iterations}")
        
        abc = ABCAlgorithm(
            y_sorted=y_sorted,
            z_sorted=z_sorted,
            pik_sorted=pik_sorted,
            var_srs_y=var_srs_y,
            var_srs_z=var_srs_z,
            M=n,
            n=n,
            case_name=f"{z_name}_N{N}_n{n}"
        )
        
        res = abc.optimize(
            colony_size=colony_size,
            max_iterations=max_iterations,
            limit=limit,
            verbose=True  # Show progress!
        )
        
        print(f"\n    Results:")
        print(f"      P_π(z)  = {res['optimal_eff_z']:7.2f}")
        print(f"      ABC(z)  = {res['best_eff_z']:7.2f}")
        print(f"      Gap     = {res['improvement_percent']:+6.2f}%")
        print(f"      Time    = {res['total_time']:6.1f}s")
        
        all_results.append({
            'z': z_name,
            'corr': actual_corr,
            'n': n,
            'N': N,
            'ppi_z': res['optimal_eff_z'],
            'abc_z': res['best_eff_z'],
            'ppi_y': res['optimal_eff_y'],
            'abc_y': res['best_eff_y'],
            'gap': res['improvement_percent'],
            'time': res['total_time'],
            'colony': colony_size,
            'iterations': max_iterations
        })

# Final table
print(f"\n\n{'='*80}")
print("FINAL SUMMARY TABLE")
print(f"{'='*80}\n")

df_res = pd.DataFrame(all_results)

for z in df_res['z'].unique():
    sub = df_res[df_res['z'] == z]
    print(f"\n{z} (ρ = {sub['corr'].iloc[0]:.3f})")
    print("-"*80)
    print(f"{'n':>5} {'P_π(z)':>8} {'ABC(z)':>8} {'Gap%':>7} {'Time(s)':>9} {'Iter':>6}")
    print("-"*80)
    for _, row in sub.iterrows():
        print(f"{row['n']:5.0f} {row['ppi_z']:8.2f} {row['abc_z']:8.2f} "
              f"{row['gap']:+7.1f} {row['time']:9.1f} {row['iterations']:6.0f}")

# Save
df_res.to_csv('abc_mu284_N281_full.csv', index=False)
print(f"\n✅ Saved: abc_mu284_N281_full.csv")

ABC ON FULL MU284 DATA (N=281)

High Corr: z = ME84
Corr(P85, ME84) = 0.988
N = 281 (FULL dataset)

  n = 5:
    ABC params: colony=200, iter=1000
 ABC ALGORITHM - ME84_N281_n5

 Configuration:
   Colony size: 200
   Max iterations: 1000
   Abandonment limit: 100

 P_π optimal (target):
   eff_z = 214.5640
   eff_y = 154.0485

 Initializing 200 food sources...
   Evaluated 4000, found 153/200 valid sources
    Initialized 153 food sources (out of 200 requested)
    Best initial: eff_z=120.4849

 ABC Optimization (1000 iterations)...

